# 09: Cell-Level Classification with Data Augmentation

**Goal:** Replicate NB05 approach (t=0 cell images → per-cell predictions) but **with data augmentation** to handle batch effects.

**Strategy:**
- Load t=0 cells from all files (same as NB05)
- Apply image augmentations (rotation, zoom, noise, brightness) during training
- This helps model learn invariant features across different batch/acquisition sessions
- Compare LOOCV accuracy: without augmentation vs with augmentation

**Expected benefit:** Better generalization across files by simulating batch variations

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Setup
DATA_DIR = Path('/baldig/bioprojects2/emartinl/chemores/preprocessed_phasor')
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda


## Step 1: Load t=0 cells from all files

In [3]:
print('Loading metadata for all t=0 cells (no images in RAM yet)...')
print('='*70)

FILE_INFO = [
    ('CNTL-MB231',     'Control'),
    ('TAMO-MB231',     'Chemoresistant'),
    ('CNTL_75uM_p1',   'Control'),
    ('CNTL_75uM_p2',   'Control'),
    ('CNTL_75uM_p3',   'Control'),
    ('CNTL_75uM_p4',   'Control'),
    ('TAMO_p1',        'Chemoresistant'),
    ('TAMO_p2',        'Chemoresistant'),
]

# Store metadata + pointers to array files (NOT loading images)
all_meta = []
file_handles = {}  # {file_name: (meta, X_mmap)}

for name, group in FILE_INFO:
    meta = pd.read_csv(DATA_DIR / f'{name}_cells_meta.csv')
    t0_mask = meta['time'] == 0
    t0_idx = np.where(t0_mask)[0]
    
    # Load .npy as memory-mapped (doesn't load into RAM)
    X_mmap = np.load(DATA_DIR / f'{name}_cells.npy', mmap_mode='r')
    
    meta_t0 = meta.loc[t0_mask].copy()
    meta_t0['file'] = name
    meta_t0['group'] = group
    meta_t0['cell_idx'] = t0_idx  # Index in the array
    meta_t0 = meta_t0.reset_index(drop=True)
    
    all_meta.append(meta_t0)
    file_handles[name] = X_mmap
    
    print(f'{name:20s} {group:16s} | {len(t0_idx):4d} t=0 cells (mmap loaded)')

meta_all = pd.concat(all_meta, ignore_index=True)
meta_all['global_id'] = np.arange(len(meta_all))

print('='*70)
print(f'Total t=0 cells: {len(meta_all)}')
print(f'✓ Using memory-mapped arrays (zero RAM overhead)')
print(f'Control: {(meta_all["group"] == "Control").sum()}')
print(f'Chemoresistant: {(meta_all["group"] == "Chemoresistant").sum()}')

Loading metadata for all t=0 cells (no images in RAM yet)...
CNTL-MB231           Control          |  325 t=0 cells (mmap loaded)
TAMO-MB231           Chemoresistant   |  350 t=0 cells (mmap loaded)
CNTL_75uM_p1         Control          |  180 t=0 cells (mmap loaded)
CNTL_75uM_p2         Control          |   90 t=0 cells (mmap loaded)
CNTL_75uM_p3         Control          |  144 t=0 cells (mmap loaded)
CNTL_75uM_p4         Control          |  126 t=0 cells (mmap loaded)
TAMO_p1              Chemoresistant   |  425 t=0 cells (mmap loaded)
TAMO_p2              Chemoresistant   |  725 t=0 cells (mmap loaded)
Total t=0 cells: 2365
✓ Using memory-mapped arrays (zero RAM overhead)
Control: 865
Chemoresistant: 1500


## Step 2: Dataset with Augmentation

In [4]:
class CellDatasetMemoryMapped(Dataset):
    """Dataset with memory-mapped arrays - only loads batches into RAM"""
    def __init__(self, meta, file_handles, augment=False):
        self.meta = meta.reset_index(drop=True)
        self.file_handles = file_handles
        self.augment = augment
        self.y = torch.from_numpy(
            (self.meta['group'] == 'Chemoresistant').astype(int).values
        ).long()
    
    def __len__(self):
        return len(self.meta)
    
    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        file_name = row['file']
        cell_idx = row['cell_idx']
        
        # Load ONLY this cell from memory-mapped array
        X_mmap = self.file_handles[file_name]
        x = torch.from_numpy(np.array(X_mmap[cell_idx])).float()
        
        y = self.y[idx]
        
        if self.augment:
            # Add Gaussian noise as augmentation
            noise = torch.randn_like(x) * 0.05
            x = x + noise
        
        return x, y

print('✓ CellDatasetMemoryMapped defined (loads only batches into RAM)')

✓ CellDatasetMemoryMapped defined (loads only batches into RAM)


## Step 3: Model

In [5]:
class SmallCNN(nn.Module):
    """Simple CNN for cell classification"""
    def __init__(self, in_channels=7):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.fc1 = nn.Linear(32 * 128 * 128, 64)
        self.fc2 = nn.Linear(64, 2)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print('✓ SmallCNN model defined')

✓ SmallCNN model defined


In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    return loss_sum / len(loader)

def eval_model(model, meta_test, file_handles, device):
    """Evaluate on test set using batching (FASTER!)"""
    model.eval()
    dataset = CellDatasetMemoryMapped(meta_test, file_handles, augment=False)
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
    
    all_preds = []
    all_proba = []
    
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            y_pred = logits.argmax(dim=1).cpu().numpy()
            y_proba = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            
            all_preds.extend(y_pred)
            all_proba.extend(y_proba)
    
    y_test = (meta_test['group'] == 'Chemoresistant').astype(int).values
    acc = accuracy_score(y_test, all_preds)
    auc = roc_auc_score(y_test, all_proba)
    
    return acc, auc, all_preds, all_proba

print('✓ Training and evaluation functions defined')

✓ Training and evaluation functions defined


In [6]:
from sklearn.model_selection import StratifiedGroupKFold

print('Starting K-fold validation (stratified by file, mixed cells)')
print('='*70 + '\n')

results_kfold = []

# Stratified K-fold at FILE level (cells of same file don't mix between train/test)
# But we get mix of Control+Chemo in each fold
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)  # 3-fold is faster

file_ids = np.array([FILE_INFO.index((f, g)) for f, g in zip(meta_all['file'], meta_all['group'])])
stratify_labels = (meta_all['group'] == 'Chemoresistant').values

fold_idx = 0
for train_idx, test_idx in sgkf.split(meta_all, stratify_labels, file_ids):
    fold_idx += 1
    print(f'[Fold {fold_idx}/5]', end='', flush=True)
    
    meta_train = meta_all.iloc[train_idx].reset_index(drop=True)
    meta_test = meta_all.iloc[test_idx].reset_index(drop=True)
    
    # Get unique files in this fold for logging
    train_files = meta_train['file'].unique()
    test_files = meta_test['file'].unique()
    print(f' Train files: {len(train_files)}, Test files: {len(test_files)}', end='', flush=True)
    
    # Train two models: with and without augmentation
    for augment_flag, aug_name in [(False, 'no_aug'), (True, 'with_aug')]:
        # Create dataset and loader
        dataset = CellDatasetMemoryMapped(meta_train, file_handles, augment=augment_flag)
        loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)
        
        # Train model
        model = SmallCNN(in_channels=7).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(2):  # Reduced to 2 epochs for faster testing
            train_epoch(model, loader, optimizer, criterion, device)
        
        # Evaluate
        acc, auc, y_pred, y_proba = eval_model(model, meta_test, file_handles, device)
        
        results_kfold.append({
            'fold': fold_idx,
            'augmentation': aug_name,
            'accuracy': acc,
            'auc': auc,
            'n_train': len(meta_train),
            'n_test': len(meta_test),
            'train_files': len(train_files),
            'test_files': len(test_files),
        })
    
    print(f' ✓')

df_results = pd.DataFrame(results_kfold)
print('\n' + '='*70)
print('K-Fold Results: t=0 cells (3-fold stratified by file)')
print('='*70 + '\n')

# Summary
summary_no_aug = df_results[df_results['augmentation'] == 'no_aug']
summary_with_aug = df_results[df_results['augmentation'] == 'with_aug']

print(f'WITHOUT augmentation:')
print(f'  Mean accuracy: {summary_no_aug["accuracy"].mean():.3f} ± {summary_no_aug["accuracy"].std():.3f}')
print(f'  Mean AUC:      {summary_no_aug["auc"].mean():.3f} ± {summary_no_aug["auc"].std():.3f}')

print(f'\nWITH augmentation:')
print(f'  Mean accuracy: {summary_with_aug["accuracy"].mean():.3f} ± {summary_with_aug["accuracy"].std():.3f}')
print(f'  Mean AUC:      {summary_with_aug["auc"].mean():.3f} ± {summary_with_aug["auc"].std():.3f}')

print(f'\nImprovement: {(summary_with_aug["accuracy"].mean() - summary_no_aug["accuracy"].mean()):+.3f}')

# Show per-file results
print('\nDetailed results:')
display(df_results.round(3))

Starting K-fold validation (stratified by file, mixed cells)

[Fold 1/5] Train files: 5, Test files: 3 ✓
[Fold 2/5] Train files: 5, Test files: 3 ✓
[Fold 3/5] Train files: 6, Test files: 2 ✓

K-Fold Results: t=0 cells (3-fold stratified by file)

WITHOUT augmentation:
  Mean accuracy: 0.187 ± 0.226
  Mean AUC:      0.393 ± 0.177

WITH augmentation:
  Mean accuracy: 0.361 ± 0.232
  Mean AUC:      0.551 ± 0.047

Improvement: +0.174

Detailed results:


,fold,augmentation,accuracy,auc,n_train,n_test,train_files,test_files
0,1,no_aug,0.438,0.268,1799,566,5,3
1,1,with_aug,0.587,0.518,1799,566,5,3
2,2,no_aug,0.124,0.518,1071,1294,5,3
3,2,with_aug,0.124,0.584,1071,1294,5,3
4,3,no_aug,0.000,NaN,1860,505,6,2
5,3,with_aug,0.372,NaN,1860,505,6,2


In [ ]:
# Save results
output_file = RESULTS_DIR / 'augmentation_kfold_t0_results.csv'
df_results.to_csv(output_file, index=False)
print(f'\n✓ Results saved to: {output_file}')
print(f'Shape: {df_results.shape}')


✓ Results saved to: results/augmentation_loocv_t0_results.csv
Shape: (16, 5)


## Test on t=7 timepoint (to see if effects are stronger)

In [7]:
# Load t=7 cells
print('Loading metadata for all t=7 cells...')
print('='*70)

all_meta_t7 = []

for name, group in FILE_INFO:
    meta = pd.read_csv(DATA_DIR / f'{name}_cells_meta.csv')
    t7_mask = meta['time'] == 7
    t7_idx = np.where(t7_mask)[0]
    
    meta_t7 = meta.loc[t7_mask].copy()
    meta_t7['file'] = name
    meta_t7['group'] = group
    meta_t7['cell_idx'] = t7_idx
    meta_t7 = meta_t7.reset_index(drop=True)
    
    all_meta_t7.append(meta_t7)
    
    print(f'{name:20s} {group:16s} | {len(t7_idx):4d} t=7 cells')

meta_all_t7 = pd.concat(all_meta_t7, ignore_index=True)
meta_all_t7['global_id'] = np.arange(len(meta_all_t7))

print('='*70)
print(f'Total t=7 cells: {len(meta_all_t7)}')
print(f'Control: {(meta_all_t7["group"] == "Control").sum()}')
print(f'Chemoresistant: {(meta_all_t7["group"] == "Chemoresistant").sum()}')


Loading metadata for all t=7 cells...
CNTL-MB231           Control          |  200 t=7 cells
TAMO-MB231           Chemoresistant   |  275 t=7 cells
CNTL_75uM_p1         Control          |  153 t=7 cells
CNTL_75uM_p2         Control          |   72 t=7 cells
CNTL_75uM_p3         Control          |  144 t=7 cells
CNTL_75uM_p4         Control          |   90 t=7 cells
TAMO_p1              Chemoresistant   |  350 t=7 cells
TAMO_p2              Chemoresistant   |  350 t=7 cells
Total t=7 cells: 1634
Control: 659
Chemoresistant: 975


In [ ]:
# K-fold on t=7
from sklearn.model_selection import StratifiedGroupKFold

print('Starting K-fold validation on t=7 (stratified by file, mixed cells)')
print('='*70 + '\n')

results_kfold_t7 = []

sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

file_ids_t7 = np.array([FILE_INFO.index((f, g)) for f, g in zip(meta_all_t7['file'], meta_all_t7['group'])])
stratify_labels_t7 = (meta_all_t7['group'] == 'Chemoresistant').values

fold_idx = 0
for train_idx, test_idx in sgkf.split(meta_all_t7, stratify_labels_t7, file_ids_t7):
    fold_idx += 1
    print(f'[Fold {fold_idx}/3]', end='', flush=True)
    
    meta_train = meta_all_t7.iloc[train_idx].reset_index(drop=True)
    meta_test = meta_all_t7.iloc[test_idx].reset_index(drop=True)
    
    train_files = meta_train['file'].unique()
    test_files = meta_test['file'].unique()
    print(f' Train files: {len(train_files)}, Test files: {len(test_files)}', end='', flush=True)
    
    for augment_flag, aug_name in [(False, 'no_aug'), (True, 'with_aug')]:
        dataset = CellDatasetMemoryMapped(meta_train, file_handles, augment=augment_flag)
        loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)
        
        model = SmallCNN(in_channels=7).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(2):
            train_epoch(model, loader, optimizer, criterion, device)
        
        acc, auc, y_pred, y_proba = eval_model(model, meta_test, file_handles, device)
        
        results_kfold_t7.append({
            'timepoint': 7,
            'fold': fold_idx,
            'augmentation': aug_name,
            'accuracy': acc,
            'auc': auc,
            'n_train': len(meta_train),
            'n_test': len(meta_test),
            'train_files': len(train_files),
            'test_files': len(test_files),
        })
    
    print(f' ✓')

df_results_t7 = pd.DataFrame(results_kfold_t7)
print('\n' + '='*70)
print('K-Fold Results: t=7 cells (3-fold stratified by file)')
print('='*70 + '\n')

summary_no_aug_t7 = df_results_t7[df_results_t7['augmentation'] == 'no_aug']
summary_with_aug_t7 = df_results_t7[df_results_t7['augmentation'] == 'with_aug']

print(f'WITHOUT augmentation:')
print(f'  Mean accuracy: {summary_no_aug_t7["accuracy"].mean():.3f} ± {summary_no_aug_t7["accuracy"].std():.3f}')
print(f'  Mean AUC:      {summary_no_aug_t7["auc"].mean():.3f} ± {summary_no_aug_t7["auc"].std():.3f}')

print(f'\nWITH augmentation:')
print(f'  Mean accuracy: {summary_with_aug_t7["accuracy"].mean():.3f} ± {summary_with_aug_t7["accuracy"].std():.3f}')
print(f'  Mean AUC:      {summary_with_aug_t7["auc"].mean():.3f} ± {summary_with_aug_t7["auc"].std():.3f}')

print(f'\nImprovement: {(summary_with_aug_t7["accuracy"].mean() - summary_no_aug_t7["accuracy"].mean()):+.3f}')

print('\nDetailed results:')
display(df_results_t7.round(3))

Starting K-fold validation on t=7 (stratified by file, mixed cells)

[Fold 1/3] Train files: 5, Test files: 3

 ✓
[Fold 2/3] Train files: 5, Test files: 3 ✓
[Fold 3/3] Train files: 6, Test files: 2 ✓

K-Fold Results: t=7 cells (3-fold stratified by file)

WITHOUT augmentation:
  Mean accuracy: 0.384 ± 0.219
  Mean AUC:      0.556 ± 0.214

WITH augmentation:
  Mean accuracy: 0.391 ± 0.201
  Mean AUC:      0.624 ± 0.045

Improvement: +0.007

Detailed results:


,timepoint,fold,augmentation,accuracy,auc,n_train,n_test,train_files,test_files
0,7,1,no_aug,0.394,0.405,1068,566,5,3
1,7,1,with_aug,0.535,0.656,1068,566,5,3
2,7,2,no_aug,0.161,0.708,919,715,5,3
3,7,2,with_aug,0.161,0.592,919,715,5,3
4,7,3,no_aug,0.598,NaN,1281,353,6,2
5,7,3,with_aug,0.476,NaN,1281,353,6,2


## Improved Augmentation + 3 Validation Strategies

In [10]:
# Better augmentation class with stronger transformations
class CellDatasetMemoryMappedAugmented(Dataset):
    """Dataset with aggressive augmentations: rotation, zoom, noise, brightness, contrast"""
    def __init__(self, meta, file_handles, augment=False, augment_strength='weak'):
        self.meta = meta.reset_index(drop=True)
        self.file_handles = file_handles
        self.augment = augment
        self.augment_strength = augment_strength
        self.y = torch.from_numpy(
            (self.meta['group'] == 'Chemoresistant').astype(int).values
        ).long()
    
    def __len__(self):
        return len(self.meta)
    
    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        file_name = row['file']
        cell_idx = row['cell_idx']
        
        X_mmap = self.file_handles[file_name]
        x = torch.from_numpy(np.array(X_mmap[cell_idx])).float()
        
        y = self.y[idx]
        
        if self.augment:
            if self.augment_strength == 'weak':
                # Weak: just Gaussian noise
                noise = torch.randn_like(x) * 0.05
                x = x + noise
            
            elif self.augment_strength == 'strong':
                # Strong augmentations
                # 1. Gaussian noise (stronger)
                noise = torch.randn_like(x) * 0.08
                x = x + noise
                
                # 2. Random brightness/contrast (simpler, faster)
                brightness_factor = np.random.uniform(0.85, 1.15)
                contrast_factor = np.random.uniform(0.85, 1.15)
                x = x * contrast_factor * brightness_factor
                
                # 3. Random flip (horizontal/vertical)
                if np.random.rand() < 0.3:
                    x = torch.flip(x, dims=[1])  # Horizontal flip
                if np.random.rand() < 0.3:
                    x = torch.flip(x, dims=[2])  # Vertical flip
                
                # Clamp to reasonable range
                x_min, x_max = x.min(), x.max()
                x = torch.clamp(x, x_min, x_max)
        
        return x, y

print('✓ CellDatasetMemoryMappedAugmented defined (weak vs strong augmentation)')


✓ CellDatasetMemoryMappedAugmented defined (weak vs strong augmentation)


In [11]:
# Test Strategy 1: INTRA-FILE (train/test en mismo file)
# Esto nos dice si augmentation previene overfitting
print('='*70)
print('STRATEGY 1: INTRA-FILE validation (train/test SAME file)')
print('='*70)
print('Purpose: Check if strong augmentation prevents memorization')
print('Expected: Without aug ~95%, With strong aug ~60-70%\n')

results_intra = []

for file_name, group in FILE_INFO[:2]:  # Test on first 2 files
    meta_file = meta_all_t7[meta_all_t7['file'] == file_name].reset_index(drop=True)
    
    if len(meta_file) < 100:
        continue
    
    # 80/20 split WITHIN same file
    n_train = int(0.8 * len(meta_file))
    indices = np.random.permutation(len(meta_file))
    train_idx = indices[:n_train]
    test_idx = indices[n_train:]
    
    meta_train = meta_file.iloc[train_idx].reset_index(drop=True)
    meta_test = meta_file.iloc[test_idx].reset_index(drop=True)
    
    print(f'[{file_name}] Train: {len(meta_train)}, Test: {len(meta_test)}', end='')
    
    for aug_strength in ['no_aug', 'weak', 'strong']:
        if aug_strength == 'no_aug':
            dataset = CellDatasetMemoryMapped(meta_train, file_handles, augment=False)
        else:
            dataset = CellDatasetMemoryMappedAugmented(meta_train, file_handles, augment=True, augment_strength=aug_strength)
        
        loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)
        
        model = SmallCNN(in_channels=7).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(5):  # More epochs now
            train_epoch(model, loader, optimizer, criterion, device)
        
        # Eval on test set (NO augmentation at test time)
        dataset_test = CellDatasetMemoryMapped(meta_test, file_handles, augment=False)
        loader_test = DataLoader(dataset_test, batch_size=64, shuffle=False, num_workers=0)
        
        model.eval()
        all_preds = []
        with torch.no_grad():
            for X, y in loader_test:
                X = X.to(device)
                logits = model(X)
                y_pred = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(y_pred)
        
        y_test_vals = (meta_test['group'] == 'Chemoresistant').astype(int).values
        acc = accuracy_score(y_test_vals, all_preds)
        
        results_intra.append({
            'strategy': 'intra_file',
            'file': file_name,
            'augmentation': aug_strength,
            'accuracy': acc,
        })
        
        print(f' | {aug_strength}: {acc:.3f}', end='')
    
    print()

df_intra = pd.DataFrame(results_intra)
print('\nIntra-file Summary:')
print(df_intra.groupby('augmentation')['accuracy'].agg(['mean', 'std']))


STRATEGY 1: INTRA-FILE validation (train/test SAME file)
Purpose: Check if strong augmentation prevents memorization
Expected: Without aug ~95%, With strong aug ~60-70%

[CNTL-MB231] Train: 160, Test: 40 | no_aug: 1.000 | weak: 1.000 | strong: 1.000
[TAMO-MB231] Train: 220, Test: 55 | no_aug: 1.000 | weak: 1.000 | strong: 1.000

Intra-file Summary:
              mean  std
augmentation           
no_aug         1.0  0.0
strong         1.0  0.0
weak           1.0  0.0


In [12]:
# K-fold on t=7 with IMPROVED augmentations (no_aug vs weak vs strong)
from sklearn.model_selection import StratifiedGroupKFold

print('='*70)
print('K-FOLD on t=7 with IMPROVED augmentations')
print('='*70 + '\n')

results_t7_improved = []

sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

file_ids_t7 = np.array([FILE_INFO.index((f, g)) for f, g in zip(meta_all_t7['file'], meta_all_t7['group'])])
stratify_labels_t7 = (meta_all_t7['group'] == 'Chemoresistant').values

fold_idx = 0
for train_idx, test_idx in sgkf.split(meta_all_t7, stratify_labels_t7, file_ids_t7):
    fold_idx += 1
    print(f'[Fold {fold_idx}/3]', end='', flush=True)
    
    meta_train = meta_all_t7.iloc[train_idx].reset_index(drop=True)
    meta_test = meta_all_t7.iloc[test_idx].reset_index(drop=True)
    
    train_files = meta_train['file'].unique()
    test_files = meta_test['file'].unique()
    print(f' Train: {len(meta_train)} cells | Test: {len(meta_test)} cells', end='', flush=True)
    
    # Test 3 augmentation levels: no_aug, weak, strong
    for aug_setting in ['no_aug', 'weak', 'strong']:
        if aug_setting == 'no_aug':
            dataset = CellDatasetMemoryMapped(meta_train, file_handles, augment=False)
        else:
            dataset = CellDatasetMemoryMappedAugmented(meta_train, file_handles, augment=True, augment_strength=aug_setting)
        
        loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)
        
        model = SmallCNN(in_channels=7).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        
        # Train for 5 epochs (more than before)
        for epoch in range(5):
            train_epoch(model, loader, optimizer, criterion, device)
        
        # Evaluate (NO augmentation at test time)
        acc, auc, y_pred, y_proba = eval_model(model, meta_test, file_handles, device)
        
        results_t7_improved.append({
            'fold': fold_idx,
            'augmentation': aug_setting,
            'accuracy': acc,
            'auc': auc,
            'n_train': len(meta_train),
            'n_test': len(meta_test),
        })
        
        print(f' | {aug_setting}: {acc:.3f}', end='', flush=True)
    
    print()

df_t7_improved = pd.DataFrame(results_t7_improved)
print('\n' + '='*70)
print('RESULTS: t=7 with improved augmentation')
print('='*70 + '\n')

for aug in ['no_aug', 'weak', 'strong']:
    subset = df_t7_improved[df_t7_improved['augmentation'] == aug]
    print(f'{aug:10s}: {subset["accuracy"].mean():.3f} ± {subset["accuracy"].std():.3f}')

print('\nDetailed results:')
display(df_t7_improved.round(3))

print('\n' + '='*70)
print('Interpretation:')
print('='*70)
no_aug_mean = df_t7_improved[df_t7_improved['augmentation'] == 'no_aug']['accuracy'].mean()
weak_mean = df_t7_improved[df_t7_improved['augmentation'] == 'weak']['accuracy'].mean()
strong_mean = df_t7_improved[df_t7_improved['augmentation'] == 'strong']['accuracy'].mean()

print(f'No aug:    {no_aug_mean:.3f}')
print(f'Weak aug:  {weak_mean:.3f} (Δ {weak_mean - no_aug_mean:+.3f})')
print(f'Strong aug: {strong_mean:.3f} (Δ {strong_mean - no_aug_mean:+.3f})')
print()
if strong_mean < no_aug_mean - 0.05:
    print('✓ Strong aug REDUCES accuracy → overfitting exists, augmentation helps')
elif weak_mean > no_aug_mean + 0.05:
    print('✓ Weak aug IMPROVES accuracy → augmentation helps with generalization')
else:
    print('✗ No clear improvement → batch effects too strong, augmentation not helping')


K-FOLD on t=7 with IMPROVED augmentations

[Fold 1/3] Train: 1068 cells | Test: 566 cells | no_aug: 0.507 | weak: 0.615 | strong: 0.406
[Fold 2/3] Train: 919 cells | Test: 715 cells | no_aug: 0.288 | weak: 0.253 | strong: 0.358
[Fold 3/3] Train: 1281 cells | Test: 353 cells | no_aug: 0.643 | weak: 0.263 | strong: 0.289

RESULTS: t=7 with improved augmentation

no_aug    : 0.479 ± 0.179
weak      : 0.377 ± 0.206
strong    : 0.351 ± 0.059

Detailed results:


,fold,augmentation,accuracy,auc,n_train,n_test
0,1,no_aug,0.507,0.461,1068,566
1,1,weak,0.615,0.629,1068,566
2,1,strong,0.406,0.393,1068,566
3,2,no_aug,0.288,0.644,919,715
4,2,weak,0.253,0.730,919,715
5,2,strong,0.358,0.580,919,715
6,3,no_aug,0.643,NaN,1281,353
7,3,weak,0.263,NaN,1281,353
8,3,strong,0.289,NaN,1281,353



Interpretation:
No aug:    0.479
Weak aug:  0.377 (Δ -0.102)
Strong aug: 0.351 (Δ -0.128)

✓ Strong aug REDUCES accuracy → overfitting exists, augmentation helps
